### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="keel_appendicitis",
    dataset_year="1991",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://sci2s.ugr.es/keel/dataset.php?cod=183", # alt https://www.kaggle.com/datasets/timrie/appendicitis
    download_description="""
We get the data from the original KEEL website.

wget https://sci2s.ugr.es/keel/dataset/data/classification/appendicitis.zip && unzip appendicitis.zip && rm appendicitis.zip
mkdir -p local-data-warehouse/keel_appendicitis && mv appendicitis.dat   local-data-warehouse/keel_appendicitis/
""",
    # References
    academic_reference_bibtex="""@book{weiss1991computer,
  title={Computer systems that learn: classification and prediction methods from statistics, neural nets, machine learning, and expert systems},
  author={Weiss, Sholom M and Kulikowski, Casimir A},
  year={1991},
  publisher={Morgan Kaufmann Publishers Inc.}
}
""",
    academic_reference_bibtex_key="weiss1991computer",
    license="None",
    data_tags=["IID"],
    curation_comments="""
The data requires no further curation.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Class",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
import arff
import re

# Clean KEEL format
input_path = dataset_mold.path / "appendicitis.dat"
clean_path =dataset_mold.path / "appendicitis.arff"

with open(input_path, "r", encoding="utf-8") as f:
    content = f.read()

# Replace things like real[0.0,1.0] → real
content = re.sub(r"(real|numeric|integer)\s*\[[^\]]*\]", r"\1", content, flags=re.IGNORECASE)

with open(clean_path, "w", encoding="utf-8") as f:
    f.write(content)

with open(clean_path) as f:
    dataset = arff.load(f)

df = pd.DataFrame(dataset["data"], columns=[attr[0] for attr in dataset["attributes"]])

print("Loaded data shape:", df.shape)

df["Class"] = df["Class"].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (106, 8)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 106
Columns: 8
Use sampling: False (sample size: 106)
Get row duplicates (staged, merged)...
Using top-7 columns for initial filtering: ['At7', 'At3', 'At6', 'At5', 'At1', 'At2', 'At4']
Rows remaining as candidates after top-7 filter: 0 (of 106)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,At1,At2,At3,At4,At5,At6,At7,Class
0,0.778,0.732,0.769,0.722,1.000,0.503,0.608,0
1,0.058,0.589,0.087,0.583,0.196,0.576,0.060,1
2,0.236,0.804,0.289,0.111,0.066,0.756,0.241,1
3,0.587,0.875,0.662,0.625,0.692,0.911,0.616,0
4,0.373,0.589,0.349,0.056,0.044,0.430,0.276,0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Class,category,0.0,0.0,2.0,"0, 1"
1,At1,float64,0.0,0.0,74.0,"0.44, 0.236, 0.413, 0.378, 0.471, 0.298, 0.187, 0.453, 0.222, 0.373"
2,At2,float64,0.0,0.0,41.0,"0.589, 0.75, 0.768, 0.857, 0.732, 0.911, 0.875, 0.786, 0.804, 0.714"
3,At3,float64,0.0,0.0,95.0,"0.482, 0.057, 0.547, 0.365, 0.544, 0.276, 0.352, 0.415, 0.466, 0.508"
4,At4,float64,0.0,0.0,39.0,"0.056, 0.0, 0.083, 0.111, 0.069, 0.097, 0.139, 0.028, 0.153, 0.181"
5,At5,float64,0.0,0.0,89.0,"0.0, 0.196, 0.113, 0.078, 0.392, 0.048, 0.029, 0.053, 0.259, 0.022"
6,At6,float64,0.0,0.0,93.0,"0.576, 0.741, 0.818, 0.743, 0.78, 0.82, 0.765, 0.754, 0.845, 0.836"
7,At7,float64,0.0,0.0,99.0,"0.428, 0.483, 0.384, 0.69, 0.436, 0.487, 0.319, 0.616, 0.241, 0.608"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
At1,106.0,0.398491,0.191452,0.0,1.0
At2,106.0,0.682104,0.207134,0.0,1.0
At3,106.0,0.415151,0.205799,0.0,1.0
At4,106.0,0.208745,0.199650,0.0,1.0
At5,106.0,0.169151,0.177329,0.0,1.0
At6,106.0,0.676349,0.218909,0.0,1.0
At7,106.0,0.375396,0.198132,0.0,1.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                    
Class  1        0     85  80.19
       2        1     21  19.81

In [8]:
# Target Distribution
target_df

,count,pct
Class,,
0,85,80.19
1,21,19.81


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7234-398e-70bc-8d43-fe053eaca337
fb7b4e5bbd3b61059aaabd9a64604b203daef200b696c290e81b65f3a9cb430c
